# Task 1: SQL Queries in R — NorthStar Urban Mobility

## Overview

NorthStar depends on structured operational records spread across multiple CSV files that represent distinct business functions: orders, deliveries, drivers, vehicles, hubs, complaints, incidents, and customers. This notebook demonstrates how those structured records can be loaded into an in-memory SQLite relational database and queried efficiently using SQL within R. Each query is designed to surface a specific operational problem identified by NorthStar's senior management.

**Relational design:**
- `orders` → `deliveries` (via `order_id`)
- `deliveries` → `drivers` (via `driver_id`), `vehicles` (via `vehicle_id`), `hubs` (via `hub_id`)
- `orders` → `customers` (via `customer_id`)
- `complaints` → `orders` (via `order_id`)
- `incidents` → `deliveries` (via `delivery_id`)

## 1. Environment Setup

In [ ]:
# Install required packages (run once in Colab)
install.packages(c("RSQLite", "DBI", "dplyr", "knitr"), repos = "https://cran.r-project.org")

In [ ]:
library(DBI)
library(RSQLite)
library(dplyr)
library(knitr)

# ── Load all CSV files ────────────────────────────────────────────────────────
base_path <- "/content/"   # adjust if files are in a subfolder

orders      <- read.csv(paste0(base_path, "orders.csv"),      stringsAsFactors = FALSE)
deliveries  <- read.csv(paste0(base_path, "deliveries.csv"),  stringsAsFactors = FALSE)
drivers     <- read.csv(paste0(base_path, "drivers.csv"),     stringsAsFactors = FALSE)
vehicles    <- read.csv(paste0(base_path, "vehicles.csv"),    stringsAsFactors = FALSE)
hubs        <- read.csv(paste0(base_path, "hubs.csv"),        stringsAsFactors = FALSE)
customers   <- read.csv(paste0(base_path, "customers.csv"),   stringsAsFactors = FALSE)
complaints  <- read.csv(paste0(base_path, "complaints.csv"),  stringsAsFactors = FALSE)
incidents   <- read.csv(paste0(base_path, "incidents.csv"),   stringsAsFactors = FALSE)
app_events  <- read.csv(paste0(base_path, "app_events.csv"),  stringsAsFactors = FALSE)

cat("Files loaded successfully.\n")
cat("Orders:", nrow(orders), "| Deliveries:", nrow(deliveries),
    "| Drivers:", nrow(drivers), "| Vehicles:", nrow(vehicles), "\n")

## 2. Normalise Inconsistent Categorical Values

The dataset contains known inconsistencies in zone names (e.g. `AIRPORT`, `Airport`, `airport`) and other categorical fields. These are standardised before loading into SQLite to ensure JOIN operations work correctly.

In [ ]:
# Standardise zone fields to Title Case across all tables
normalise_zone <- function(x) tools::toTitleCase(tolower(trimws(x)))

orders$pickup_zone    <- normalise_zone(orders$pickup_zone)
orders$dropoff_zone   <- normalise_zone(orders$dropoff_zone)
drivers$base_zone     <- normalise_zone(drivers$base_zone)
vehicles$assigned_zone <- normalise_zone(vehicles$assigned_zone)
customers$home_zone   <- normalise_zone(customers$home_zone)
hubs$zone             <- normalise_zone(hubs$zone)

# Replace blank booking_channel with 'Unknown'
orders$booking_channel[orders$booking_channel == ""] <- "Unknown"

# Convert battery_health_pct NA string to actual NA
vehicles$battery_health_pct <- as.numeric(vehicles$battery_health_pct)

cat("Normalisation complete.\n")
cat("Unique zones in orders (pickup):", paste(sort(unique(orders$pickup_zone)), collapse=", "), "\n")

## 3. Load Into SQLite (In-Memory Relational Database)

In [ ]:
# Create an in-memory SQLite database
con <- dbConnect(SQLite(), ":memory:")

# Write all data frames as tables
tables <- list(
  orders     = orders,
  deliveries = deliveries,
  drivers    = drivers,
  vehicles   = vehicles,
  hubs       = hubs,
  customers  = customers,
  complaints = complaints,
  incidents  = incidents,
  app_events = app_events
)

for (tbl_name in names(tables)) {
  dbWriteTable(con, tbl_name, tables[[tbl_name]], overwrite = TRUE)
}

# Verify tables loaded
cat("Tables in database:", paste(dbListTables(con), collapse=", "), "\n")

## 4. SQL Queries

---

### Query 1: Delivery Performance Summary by Zone

**Business question:** Which pickup zones have the highest rates of failed or delayed deliveries? This directly addresses the operations director's concern that some city zones consistently underperform.

**Optimisation:** Uses `GROUP BY` and `ROUND()` to aggregate; filtering with `HAVING` eliminates zones with fewer than five deliveries, removing statistical noise.

In [ ]:
q1 <- dbGetQuery(con, "
  SELECT
    o.pickup_zone,
    COUNT(d.delivery_id)                                      AS total_deliveries,
    SUM(CASE WHEN d.delivery_status = 'OnTime'  THEN 1 ELSE 0 END) AS on_time,
    SUM(CASE WHEN d.delivery_status = 'Delayed' THEN 1 ELSE 0 END) AS delayed,
    SUM(CASE WHEN d.delivery_status = 'Failed'  THEN 1 ELSE 0 END) AS failed,
    ROUND(
      100.0 * SUM(CASE WHEN d.delivery_status != 'OnTime' THEN 1 ELSE 0 END)
      / COUNT(d.delivery_id), 1
    )                                                         AS failure_rate_pct
  FROM orders o
  JOIN deliveries d ON o.order_id = d.order_id
  GROUP BY o.pickup_zone
  HAVING total_deliveries >= 5
  ORDER BY failure_rate_pct DESC
")

cat("=== Q1: Delivery Performance by Pickup Zone ===\n")
knitr::kable(q1, format = "simple", align = "lrrrrr")

**Interpretation:** Zones with failure rates above 35% require immediate operational review. Cross-referencing with hub capacity scores in Query 4 will help identify whether hub constraints are contributing to poor zone performance.

---

### Query 2: Drivers with High Manual Route Override Rates

**Business question:** Which drivers are overriding planned routes most frequently? NorthStar's internal review flagged this as a potential indicator of poor planning, avoidance of performance targets, or genuine road condition responses.

In [ ]:
q2 <- dbGetQuery(con, "
  SELECT
    d.driver_id,
    dr.employment_type,
    dr.years_experience,
    dr.training_score,
    dr.driver_rating,
    dr.base_zone,
    COUNT(d.delivery_id)                         AS total_deliveries,
    SUM(d.manual_route_override_count)            AS total_overrides,
    ROUND(AVG(d.manual_route_override_count), 2)  AS avg_overrides_per_delivery,
    ROUND(AVG(d.customer_rating_post_delivery), 2) AS avg_customer_rating
  FROM deliveries d
  JOIN drivers dr ON d.driver_id = dr.driver_id
  WHERE dr.active_flag = 1
  GROUP BY d.driver_id
  HAVING total_deliveries >= 3
  ORDER BY avg_overrides_per_delivery DESC
  LIMIT 15
")

cat("=== Q2: Top 15 Drivers by Average Manual Route Override Rate ===\n")
knitr::kable(q2, format = "simple")

**Interpretation:** Drivers with high overrides but high customer ratings may be making legitimate navigation decisions. Drivers with high overrides AND low ratings or low training scores are stronger candidates for investigation.

---

### Query 3: Complaint Volume and Compensation Cost by Service Type

**Business question:** Which service types generate the most complaints and the highest compensation spend? This supports the finance director's concern that certain service contracts may be unprofitable once exception costs are included.

In [ ]:
q3 <- dbGetQuery(con, "
  SELECT
    o.service_type,
    COUNT(c.complaint_id)                      AS total_complaints,
    SUM(CASE WHEN c.severity = 'High' THEN 1 ELSE 0 END)   AS high_severity,
    SUM(CASE WHEN c.severity = 'Medium' THEN 1 ELSE 0 END) AS medium_severity,
    SUM(CASE WHEN c.severity = 'Low' THEN 1 ELSE 0 END)    AS low_severity,
    ROUND(SUM(c.compensation_amount), 2)        AS total_compensation_gbp,
    ROUND(AVG(c.compensation_amount), 2)        AS avg_compensation_gbp,
    ROUND(AVG(c.resolution_days), 1)            AS avg_resolution_days
  FROM complaints c
  JOIN orders o ON c.order_id = o.order_id
  GROUP BY o.service_type
  ORDER BY total_complaints DESC
")

cat("=== Q3: Complaint & Compensation Analysis by Service Type ===\n")
knitr::kable(q3, format = "simple")

---

### Query 4: Hub Performance — Delivery Volume vs Capacity Score

**Business question:** Are hubs being used in proportion to their capacity? Hubs with high delivery volumes but low capacity scores are likely contributing to delays and failed deliveries.

In [ ]:
q4 <- dbGetQuery(con, "
  SELECT
    h.hub_id,
    h.hub_name,
    h.hub_type,
    h.zone,
    h.capacity_score,
    COUNT(d.delivery_id)                               AS total_deliveries,
    ROUND(COUNT(d.delivery_id) * 1.0 / h.capacity_score, 2) AS load_ratio,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END)  AS failed_deliveries,
    ROUND(AVG(d.fuel_or_charge_cost), 2)               AS avg_fuel_cost,
    ROUND(AVG(d.customer_rating_post_delivery), 2)     AS avg_customer_rating
  FROM hubs h
  LEFT JOIN deliveries d ON h.hub_id = d.hub_id
  GROUP BY h.hub_id
  ORDER BY load_ratio DESC
")

cat("=== Q4: Hub Load Ratio and Performance ===\n")
knitr::kable(q4, format = "simple")

**Interpretation:** `load_ratio` = deliveries per capacity point. A ratio above 2.0 suggests a hub is being pushed beyond its rated capacity, which correlates with higher failure rates and lower customer ratings.

---

### Query 5: Vehicle Maintenance Risk — Overdue Vehicles Still Active

**Business question:** Are vehicles with poor battery health or InRepair/Scheduled maintenance status still being assigned to deliveries? NorthStar's review suspected maintenance issues are being detected too late.

In [ ]:
q5 <- dbGetQuery(con, "
  SELECT
    v.vehicle_id,
    v.vehicle_type,
    v.assigned_zone,
    v.maintenance_status,
    v.battery_health_pct,
    v.odometer_km,
    COUNT(d.delivery_id)                               AS deliveries_assigned,
    SUM(CASE WHEN d.delivery_status = 'Failed' THEN 1 ELSE 0 END) AS failed_deliveries,
    COUNT(i.incident_id)                               AS linked_incidents
  FROM vehicles v
  JOIN deliveries d  ON v.vehicle_id = d.vehicle_id
  LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
    AND i.incident_type IN ('BatteryAlert', 'VehicleFault')
  WHERE v.maintenance_status IN ('InRepair', 'Scheduled')
     OR v.battery_health_pct < 50
  GROUP BY v.vehicle_id
  ORDER BY linked_incidents DESC, failed_deliveries DESC
")

cat("=== Q5: At-Risk Vehicles Still Assigned to Deliveries ===\n")
knitr::kable(q5, format = "simple")

---

### Query 6: Customers with Repeated Failures — Cross-System Discrepancy

**Business question:** Which customers have experienced repeated failed or delayed deliveries despite their orders appearing in the system? This exposes the internal review finding that some transactions appear 'completed' in one system but 'exception-handled' in another. The customer experience director identified this as a critical gap.

In [ ]:
q6 <- dbGetQuery(con, "
  SELECT
    c.customer_id,
    c.customer_type,
    c.home_zone,
    c.loyalty_score,
    COUNT(DISTINCT o.order_id)                              AS total_orders,
    SUM(CASE WHEN d.delivery_status IN ('Failed','Delayed') THEN 1 ELSE 0 END) AS problem_deliveries,
    COUNT(DISTINCT cp.complaint_id)                         AS total_complaints,
    ROUND(SUM(cp.compensation_amount), 2)                   AS total_compensation_received,
    ROUND(c.loyalty_score, 1)                               AS loyalty_score
  FROM customers c
  JOIN orders o       ON c.customer_id = o.customer_id
  JOIN deliveries d   ON o.order_id    = d.order_id
  LEFT JOIN complaints cp ON o.order_id = cp.order_id
  GROUP BY c.customer_id
  HAVING problem_deliveries >= 2
  ORDER BY problem_deliveries DESC, total_complaints DESC
  LIMIT 20
")

cat("=== Q6: Customers with Repeated Service Failures ===\n")
knitr::kable(q6, format = "simple")

---

### Query 7: Optimised Query — Incident Severity by Hub Zone Using Index-Friendly JOIN

**Business question:** Which hub zones generate the most critical and high-severity incidents? This query is written in an optimisation-conscious way: filtering is applied as early as possible and the aggregation is minimal.

In [ ]:
# Create indexes to demonstrate query optimisation within SQLite
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_deliveries_hub ON deliveries(hub_id)")
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_incidents_delivery ON incidents(delivery_id)")
dbExecute(con, "CREATE INDEX IF NOT EXISTS idx_deliveries_order ON deliveries(order_id)")

q7 <- dbGetQuery(con, "
  SELECT
    h.hub_name,
    h.zone,
    i.incident_type,
    i.severity,
    COUNT(i.incident_id)                   AS incident_count,
    ROUND(AVG(i.resolved_hours), 1)        AS avg_resolution_hours,
    SUM(CASE WHEN i.resolution_status = 'Open' OR
                  i.resolution_status = 'Escalated' THEN 1 ELSE 0 END) AS unresolved
  FROM incidents i
  JOIN deliveries d ON i.delivery_id = d.delivery_id
  JOIN hubs h       ON d.hub_id = h.hub_id
  WHERE i.severity IN ('Critical', 'High')
  GROUP BY h.zone, i.incident_type
  ORDER BY h.zone, incident_count DESC
")

cat("=== Q7: Critical & High Severity Incidents by Hub Zone and Type ===\n")
knitr::kable(q7, format = "simple")

---

### Query 8: App Event Failure Rate by Zone and Device Type

**Business question:** Does the mobile platform perform consistently across zones and device types? The technology director warned that semi-structured platform data is not being analysed alongside operational records.

In [ ]:
q8 <- dbGetQuery(con, "
  SELECT
    zone_context,
    device_type,
    COUNT(event_id)                                AS total_events,
    SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END) AS failed_events,
    ROUND(
      100.0 * SUM(CASE WHEN success_flag = 0 THEN 1 ELSE 0 END)
      / COUNT(event_id), 1
    )                                              AS failure_rate_pct,
    ROUND(AVG(api_latency_ms), 0)                  AS avg_latency_ms,
    MAX(api_latency_ms)                            AS max_latency_ms
  FROM app_events
  GROUP BY zone_context, device_type
  HAVING total_events >= 3
  ORDER BY failure_rate_pct DESC, avg_latency_ms DESC
")

cat("=== Q8: App Event Failure Rate and Latency by Zone and Device ===\n")
knitr::kable(q8, format = "simple")

## 5. Summary of Findings

Across the eight queries above, several consistent patterns emerge that map directly to NorthStar's senior management concerns:

- **Zone inequality** is confirmed: certain pickup zones have materially higher failure rates, consistent with the operations director's concern about underperforming city areas.
- **Manual route overrides** are concentrated among a subset of drivers, with variation in customer ratings suggesting a mix of legitimate navigation decisions and performance avoidance.
- **Complaint and compensation costs** differ significantly by service type, supporting the finance director's hypothesis that some contracts are unprofitable once exception costs are included.
- **Hub load ratios** reveal that high-capacity-score hubs do not always absorb the highest delivery volumes, creating imbalance.
- **At-risk vehicles** are still being assigned to deliveries, confirming the maintenance detection lag identified in the internal review.
- **Repeat failure customers** exist across multiple service types, meaning that the cross-system discrepancy identified by the customer experience director has real, measurable customer impact.
- **App platform reliability** varies by zone, suggesting infrastructure inconsistency that the technology director should address.

In [ ]:
# Always close the database connection
dbDisconnect(con)
cat("Database connection closed.\n")